# DWS Water Quality Data Cleaning & Matching

**Goal**: Take the already-extracted DWS water quality database CSV and clean/match it to the EY Datathon train/test set — keeping **all available columns** (not just TAL, EC, PO4_P).

**Pipeline** (mirrors `04_dws_scraper.ipynb` logic):
1. Load & inspect the DWS extracted database
2. Numeric coercion & null handling
3. Date parsing & filtering (2010–2016)
4. Spatial matching — find nearest DWS station per train/test location
5. Temporal matching — find closest measurement date per matched station
6. Unit conversions (EC ×10, PO4_P ×1000)
7. Export cleaned, matched dataset with ALL columns

**Input files** (adjust paths below):
- `dws_water_quality_db.csv` — the full extracted DWS database
- `train.csv` or `testing_all.csv` — the EY Datathon target file (Latitude, Longitude, Sample Date)

## 0. Setup & Configuration

In [21]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURE THESE PATHS TO YOUR LOCAL FILES
# ============================================================
DWS_DB_PATH = 'data/dws_water_quality_db.csv'       # Full extracted DWS database
TARGET_PATH = 'data/archive/testing_ALL.csv'  # EY train/test file
STATIONS_PATH = 'dws_stations_all.csv'          # Station metadata (optional, for coordinates)
OUTPUT_PATH = 'data/archive/dws_test.csv'  # Output file

# Date filter range (training period + buffer)
DATE_START = '2010-01-01'
DATE_END   = '2016-12-31'

print('Setup complete.')

Setup complete.


## 1. Load & Inspect the DWS Database

In [22]:
# Load the full DWS database
dws_db = pd.read_csv(DWS_DB_PATH)

print(f'Shape: {dws_db.shape}')
print(f'\nColumns ({len(dws_db.columns)}):')
for col in dws_db.columns:
    print(f'  {col}: {dws_db[col].dtype} | non-null: {dws_db[col].notna().sum():,} ({dws_db[col].notna().mean()*100:.1f}%)')

print(f'\nFirst 3 rows:')
dws_db.head(3)

Shape: (1115095, 15)

Columns (15):
  station_id: object | non-null: 1,115,095 (100.0%)
  date_time: object | non-null: 1,085,583 (97.4%)
  TAL: float64 | non-null: 690,929 (62.0%)
  EC: float64 | non-null: 1,002,453 (89.9%)
  PO4_P: float64 | non-null: 860,113 (77.1%)
  pH: float64 | non-null: 954,382 (85.6%)
  Ca: float64 | non-null: 689,786 (61.9%)
  Mg: float64 | non-null: 696,044 (62.4%)
  Na: float64 | non-null: 695,667 (62.4%)
  Cl: float64 | non-null: 736,829 (66.1%)
  SO4: float64 | non-null: 742,378 (66.6%)
  Station: object | non-null: 895,036 (80.3%)
  P_Tot: float64 | non-null: 172,995 (15.5%)
  latitude: float64 | non-null: 1,115,095 (100.0%)
  longitude: float64 | non-null: 1,115,095 (100.0%)

First 3 rows:


,station_id,date_time,TAL,EC,PO4_P,pH,Ca,Mg,Na,Cl,SO4,Station,P_Tot,latitude,longitude
0,A10_189482,2007-08-20 08:45:00,338.926,63.0,0.014,8.151,69.656,42.680,4.135,6.557,14.958,A3NGOT-PUANE,NaN,-25.42919,25.86717
1,A10_189482,2008-04-11 08:20:00,316.744,66.7,0.019,8.613,85.019,48.629,5.294,7.550,70.636,A3NGOT-PUANE,NaN,-25.42919,25.86717
2,A21_100000752,2002-06-18 12:00:00,137.000,65.0,0.050,7.800,48.000,21.500,NaN,NaN,95.000,A21,NaN,-25.83053,28.11831


In [23]:
# Load the target file (train or test)
target = pd.read_csv(TARGET_PATH)

print(f'Target shape: {target.shape}')
print(f'Columns: {list(target.columns)}')
target.head(3)

Target shape: (200, 55)
Columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'pet', '_merge_terra', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', '_merge_landsat', 'Impute_Method', 'geometry', 'STAT_ID', 'sc', 'ss', 'su', 'mt', 'va', 'vb', 'vi', 'pa', 'pb', 'pi', 'GLC_Artificial', 'GLC_Managed', 'GLC_Water', 'GLC_Aquatic_Veg', 'GLC_PERC_COV', 'Popdens_00', 'Soil_pH', 'SOC', 'Soil_wetness', 'dist_m', 'dist_km', 'Latitude_glorich', 'Longitude_glorich', 'date', 'Alkalinity', 'Cl', 'DIP', 'SO4', 'SpecCond25C', 'pH', 'Alkalinity_reliability', 'Cl_reliability', 'DIP_reliability', 'SO4_reliability', 'SpecCond25C_reliability', 'pH_reliability', 'date_diff_days']


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,_merge_terra,nir,green,...,SO4,SpecCond25C,pH,Alkalinity_reliability,Cl_reliability,DIP_reliability,SO4_reliability,SpecCond25C_reliability,pH_reliability,date_diff_days
0,-32.04333,27.82278,2014-09-01,NaN,NaN,NaN,161.90001,both,15229.0,12868.0,...,95.112905,24.976744,6.467708,1.0,1.0,1.0,1.0,1.0,0.7,0.0
1,-33.32917,26.07750,2015-09-16,NaN,NaN,NaN,177.60000,both,13065.0,10053.0,...,1504.706000,341.175105,8.373036,1.0,1.0,1.0,0.4,1.0,1.0,1.0
2,-32.99164,27.64003,2015-05-07,NaN,NaN,NaN,158.40001,both,16221.0,9304.5,...,76.263526,76.000000,7.047684,1.0,0.4,1.0,0.7,0.4,0.7,6.0


## 2. Numeric Coercion & Null Handling

Force all chemical measurement columns to numeric, coercing errors to NaN.

In [24]:
# Define ALL chemical/measurement columns to coerce
NUMERIC_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4', 'P_Tot']

# Only coerce columns that exist in the dataframe
existing_numeric = [c for c in NUMERIC_COLS if c in dws_db.columns]
missing_numeric = [c for c in NUMERIC_COLS if c not in dws_db.columns]

if missing_numeric:
    print(f'WARNING: These expected columns are missing from the database: {missing_numeric}')

# Coerce to numeric
for col in existing_numeric:
    before_null = dws_db[col].isna().sum()
    dws_db[col] = pd.to_numeric(dws_db[col], errors='coerce')
    after_null = dws_db[col].isna().sum()
    new_nulls = after_null - before_null
    if new_nulls > 0:
        print(f'  {col}: {new_nulls} non-numeric values coerced to NaN')

# Also coerce lat/lon
for col in ['latitude', 'longitude']:
    if col in dws_db.columns:
        dws_db[col] = pd.to_numeric(dws_db[col], errors='coerce')

print(f'\nNull summary after coercion:')
for col in existing_numeric:
    pct = dws_db[col].isna().mean() * 100
    print(f'  {col}: {dws_db[col].isna().sum():,} nulls ({pct:.1f}%)')

print(f'\nDescriptive stats:')
dws_db[existing_numeric].describe().round(3)


Null summary after coercion:
  TAL: 424,166 nulls (38.0%)
  EC: 112,642 nulls (10.1%)
  PO4_P: 254,982 nulls (22.9%)
  pH: 160,713 nulls (14.4%)
  Ca: 425,309 nulls (38.1%)
  Mg: 419,051 nulls (37.6%)
  Na: 419,428 nulls (37.6%)
  Cl: 378,266 nulls (33.9%)
  SO4: 372,717 nulls (33.4%)
  P_Tot: 942,100 nulls (84.5%)

Descriptive stats:


,TAL,EC,PO4_P,pH,Ca,Mg,Na,Cl,SO4,P_Tot
count,690929.000,1002453.000,860113.000,954382.000,689786.000,696044.000,695667.000,736829.000,742378.000,172995.000
mean,106.751,162.814,1.419,7.603,35.857,28.432,124.386,172.908,104.783,0.401
std,100.067,2452.169,96.312,0.861,98.717,108.263,829.964,1177.281,346.674,4.692
min,0.000,0.005,0.000,0.010,0.005,0.000,0.000,0.000,0.000,0.000
25%,35.400,17.100,0.012,7.290,8.616,5.075,9.133,9.144,7.300,0.035
50%,87.400,41.600,0.032,7.750,22.000,12.200,24.880,25.549,22.438,0.083
75%,146.091,78.000,0.142,8.150,40.200,24.100,60.076,61.901,76.200,0.253
max,6121.200,2000000.000,16860.000,18.220,37662.000,34585.000,206862.000,134075.000,36582.300,1450.000


## 3. Date Parsing & Filtering (2010–2016)

In [25]:
# Parse dates in DWS database
dws_db['date_time'] = pd.to_datetime(dws_db['date_time'], errors='coerce')

unparsed = dws_db['date_time'].isna().sum()
print(f'Date parsing: {unparsed} rows could not be parsed ({unparsed/len(dws_db)*100:.2f}%)')
print(f'Full date range: {dws_db["date_time"].min()} to {dws_db["date_time"].max()}')

# Filter to the competition period + buffer
dws_period = dws_db[
    (dws_db['date_time'] >= DATE_START) &
    (dws_db['date_time'] <= DATE_END)
].copy()

print(f'\nAfter filtering to {DATE_START} — {DATE_END}:')
print(f'  Records: {len(dws_period):,} (was {len(dws_db):,})')
print(f'  Unique stations: {dws_period["station_id"].nunique()}')

# Parse dates in target file
target['Sample Date'] = pd.to_datetime(target['Sample Date'], dayfirst=True, errors='coerce')
print(f'\nTarget date range: {target["Sample Date"].min()} to {target["Sample Date"].max()}')

Date parsing: 29512 rows could not be parsed (2.65%)
Full date range: 1960-02-23 12:15:00 to 2025-09-03 12:04:00

After filtering to 2010-01-01 — 2016-12-31:
  Records: 168,999 (was 1,115,095)
  Unique stations: 3844

Target date range: 2011-02-06 00:00:00 to 2015-12-11 00:00:00


## 4. Spatial Matching — Nearest DWS Station per Target Location

Uses the same `cdist` approach as the scraper: compute Euclidean distance on lat/lon × 111 for rough km.

In [26]:
# Get unique target locations
target_locs = (
    target
    .groupby(['Latitude', 'Longitude'])
    .size()
    .reset_index(name='n_samples')
)
print(f'Unique target locations: {len(target_locs)}')

# Get unique DWS station coordinates
# Try from the period-filtered data first (stations with data in our window)
dws_coords = (
    dws_period
    .groupby('station_id')[['latitude', 'longitude']]
    .first()
    .reset_index()
    .dropna(subset=['latitude', 'longitude'])
)
print(f'DWS stations with data in period: {len(dws_coords)}')

# Compute distance matrix (target_locs × dws_stations)
dist_matrix = cdist(
    target_locs[['Latitude', 'Longitude']].values,
    dws_coords[['latitude', 'longitude']].values
) * 111  # rough conversion to km

# For each target location, find the nearest DWS station
target_locs['best_dws'] = dws_coords['station_id'].values[dist_matrix.argmin(axis=1)]
target_locs['best_dist_km'] = dist_matrix.min(axis=1)

print(f'\nDistance to nearest DWS station:')
print(target_locs['best_dist_km'].describe().round(4))

# Merge back to all target rows
target_matched = target.merge(
    target_locs[['Latitude', 'Longitude', 'best_dws', 'best_dist_km']],
    on=['Latitude', 'Longitude']
)
print(f'\nTarget rows matched: {len(target_matched)}')

Unique target locations: 24
DWS stations with data in period: 3844

Distance to nearest DWS station:
count    24.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: best_dist_km, dtype: float64

Target rows matched: 200


## 5. Temporal Matching — Closest Date per Matched Station

For each target row, find the DWS measurement from the matched station that is closest in time.  
**Key difference from original scraper**: we pull ALL available columns, not just TAL/EC/PO4_P.

In [27]:
# Columns to carry over from DWS (all chemical + metadata)
DWS_CARRY_COLS = [c for c in existing_numeric if c in dws_period.columns]
# Also carry Station name if available
if 'Station' in dws_period.columns:
    DWS_CARRY_COLS.append('Station')

print(f'Columns to carry from DWS: {DWS_CARRY_COLS}')
print(f'Matching {len(target_matched)} target rows to DWS records...\n')

# Pre-filter: drop all rows with NaT date_time from DWS period
dws_period = dws_period.dropna(subset=['date_time']).copy()
print(f'DWS records with valid dates: {len(dws_period):,}')

results = []
skipped = 0

for station_id, group in tqdm(target_matched.groupby('best_dws')):
    sdata = dws_period[dws_period['station_id'] == station_id].sort_values('date_time')
    
    if len(sdata) == 0:
        for idx in group.index:
            results.append({'test_idx': idx})
            skipped += 1
        continue
    
    # Pre-convert station dates to numpy array for fast comparison
    station_dates = sdata['date_time'].values  # numpy datetime64 array
    
    for idx, row in group.iterrows():
        sample_date = row['Sample Date']
        
        # Skip if target date is NaT
        if pd.isna(sample_date):
            results.append({'test_idx': idx})
            skipped += 1
            continue
        
        try:
            # Compute diffs using numpy (avoids pandas idxmin issues)
            diffs = np.abs(station_dates - np.datetime64(sample_date))
            best_pos = np.argmin(diffs)
            best = sdata.iloc[best_pos]
            
            result_row = {
                'test_idx': idx,
                'dws_station_id': station_id,
                'dws_date': best['date_time'],
                'dist_km': row['best_dist_km'],
                'days_diff': int(diffs[best_pos] / np.timedelta64(1, 'D')),
            }
            
            # Carry over ALL chemical columns with dws_ prefix
            for col in DWS_CARRY_COLS:
                result_row[f'dws_{col}'] = best.get(col, np.nan)
            
            results.append(result_row)
        
        except Exception as e:
            results.append({'test_idx': idx})
            skipped += 1

matches_df = pd.DataFrame(results)
print(f'\nMatched: {matches_df["dws_station_id"].notna().sum()} / {len(matches_df)}')
print(f'Skipped: {skipped}')
print(f'\nDays difference stats:')
print(matches_df['days_diff'].describe().round(1))

Columns to carry from DWS: ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4', 'P_Tot', 'Station']
Matching 200 target rows to DWS records...

DWS records with valid dates: 168,999


100%|██████████| 24/24 [00:00<00:00, 34.86it/s]


Matched: 104 / 200
Skipped: 96

Days difference stats:
count    104.0
mean      10.1
std       12.3
min        0.0
25%        2.0
50%        6.0
75%       12.2
max       63.0
Name: days_diff, dtype: float64


In [28]:
# Coverage check: how many non-null values per DWS column?
print('Coverage per DWS column:')
dws_cols = [c for c in matches_df.columns if c.startswith('dws_') and c != 'dws_station_id' and c != 'dws_date']
for col in dws_cols:
    n_filled = matches_df[col].notna().sum()
    print(f'  {col}: {n_filled}/{len(matches_df)} ({n_filled/len(matches_df)*100:.1f}%)')

Coverage per DWS column:
  dws_TAL: 103/200 (51.5%)
  dws_EC: 103/200 (51.5%)
  dws_PO4_P: 103/200 (51.5%)
  dws_pH: 104/200 (52.0%)
  dws_Ca: 102/200 (51.0%)
  dws_Mg: 101/200 (50.5%)
  dws_Na: 66/200 (33.0%)
  dws_Cl: 103/200 (51.5%)
  dws_SO4: 97/200 (48.5%)
  dws_P_Tot: 9/200 (4.5%)
  dws_Station: 104/200 (52.0%)


## 6. Unit Conversions

Apply the same conversions as the scraper:
- **TAL**: stays as-is (mg/L CaCO₃ in both DWS and competition)
- **EC**: DWS is in mS/m → multiply by 10 to get µS/cm
- **PO4_P**: DWS is in mg/L → multiply by 1000 to get µg/L

In [29]:
# Apply unit conversions
if 'dws_EC' in matches_df.columns:
    matches_df['dws_EC_uScm'] = matches_df['dws_EC'] * 10
    print(f'EC converted: mS/m → µS/cm (×10)')
    print(f'  Range: [{matches_df["dws_EC_uScm"].min():.1f}, {matches_df["dws_EC_uScm"].max():.1f}]')

if 'dws_PO4_P' in matches_df.columns:
    matches_df['dws_PO4_P_ugL'] = matches_df['dws_PO4_P'] * 1000
    print(f'PO4_P converted: mg/L → µg/L (×1000)')
    print(f'  Range: [{matches_df["dws_PO4_P_ugL"].min():.1f}, {matches_df["dws_PO4_P_ugL"].max():.1f}]')

if 'dws_TAL' in matches_df.columns:
    print(f'TAL: no conversion needed (already mg/L CaCO₃)')
    print(f'  Range: [{matches_df["dws_TAL"].min():.1f}, {matches_df["dws_TAL"].max():.1f}]')

# Scale check against expected training ranges
print(f'\n--- SCALE CHECK (compare with training data) ---')
print(f'  Expected TAL range: ~[5, 362], mean ~119')
print(f'  Expected EC range:  ~[15, 1506], mean ~485 (µS/cm)')
print(f'  Expected DRP range: ~[5, 195], mean ~44 (µg/L)')

EC converted: mS/m → µS/cm (×10)
  Range: [99.1, 5000.0]
PO4_P converted: mg/L → µg/L (×1000)
  Range: [5.0, 1156.0]
TAL: no conversion needed (already mg/L CaCO₃)
  Range: [9.0, 514.2]

--- SCALE CHECK (compare with training data) ---
  Expected TAL range: ~[5, 362], mean ~119
  Expected EC range:  ~[15, 1506], mean ~485 (µS/cm)
  Expected DRP range: ~[5, 195], mean ~44 (µg/L)


## 7. Combine & Export

Join the matched DWS columns back to the original target dataframe and save.

In [30]:
# Align matches_df index to target
matches_df = matches_df.sort_values('test_idx').reset_index(drop=True)

# Build final output: original target columns + all DWS matched columns
final = target.copy()

# Add match metadata
final['dws_station_id'] = matches_df['dws_station_id'].values
final['dws_date'] = matches_df['dws_date'].values
final['dist_km'] = matches_df['dist_km'].values
final['days_diff'] = matches_df['days_diff'].values

# Add the 3 competition-ready columns (with unit conversions applied)
if 'dws_TAL' in matches_df.columns:
    final['Total Alkalinity (DWS)'] = matches_df['dws_TAL'].values
if 'dws_EC_uScm' in matches_df.columns:
    final['Electrical Conductance (DWS)'] = matches_df['dws_EC_uScm'].values
if 'dws_PO4_P_ugL' in matches_df.columns:
    final['Dissolved Reactive Phosphorus (DWS)'] = matches_df['dws_PO4_P_ugL'].values

# Add ALL extra chemical columns (raw DWS units)
extra_cols = ['dws_pH', 'dws_Ca', 'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_P_Tot', 'dws_Station']
for col in extra_cols:
    if col in matches_df.columns:
        final[col] = matches_df[col].values

print(f'Final dataset shape: {final.shape}')
print(f'\nColumns:')
for col in final.columns:
    print(f'  {col}')

final.head()

Final dataset shape: (200, 69)

Columns:
  Latitude
  Longitude
  Sample Date
  Total Alkalinity
  Electrical Conductance
  Dissolved Reactive Phosphorus
  pet
  _merge_terra
  nir
  green
  swir16
  swir22
  NDMI
  MNDWI
  _merge_landsat
  Impute_Method
  geometry
  STAT_ID
  sc
  ss
  su
  mt
  va
  vb
  vi
  pa
  pb
  pi
  GLC_Artificial
  GLC_Managed
  GLC_Water
  GLC_Aquatic_Veg
  GLC_PERC_COV
  Popdens_00
  Soil_pH
  SOC
  Soil_wetness
  dist_m
  dist_km
  Latitude_glorich
  Longitude_glorich
  date
  Alkalinity
  Cl
  DIP
  SO4
  SpecCond25C
  pH
  Alkalinity_reliability
  Cl_reliability
  DIP_reliability
  SO4_reliability
  SpecCond25C_reliability
  pH_reliability
  date_diff_days
  dws_station_id
  dws_date
  days_diff
  Total Alkalinity (DWS)
  Electrical Conductance (DWS)
  Dissolved Reactive Phosphorus (DWS)
  dws_pH
  dws_Ca
  dws_Mg
  dws_Na
  dws_Cl
  dws_SO4
  dws_P_Tot
  dws_Station


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,_merge_terra,nir,green,...,Electrical Conductance (DWS),Dissolved Reactive Phosphorus (DWS),dws_pH,dws_Ca,dws_Mg,dws_Na,dws_Cl,dws_SO4,dws_P_Tot,dws_Station
0,-32.04333,27.82278,2014-01-09,NaN,NaN,NaN,161.90001,both,15229.0,12868.0,...,111.60,10.0,7.554,6.295,3.134,NaN,6.156,1.500,NaN,S5H002Q01
1,-33.32917,26.07750,NaT,NaN,NaN,NaN,177.60000,both,13065.0,10053.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.99164,27.64003,2015-07-05,NaN,NaN,NaN,158.40001,both,16221.0,9304.5,...,403.00,72.0,8.116,14.828,NaN,50.370,62.200,14.432,NaN,R2H027
3,-34.09639,24.43917,2012-07-02,NaN,NaN,NaN,130.00000,both,14411.0,8184.0,...,424.00,5.0,6.363,4.615,7.082,50.566,107.851,16.138,NaN,K8H005Q01
4,-32.00056,28.58167,2014-01-10,NaN,NaN,NaN,152.50000,both,9125.0,11100.5,...,113.02,5.0,7.713,15.372,3.809,NaN,8.250,1.500,NaN,T1H015Q01


In [31]:
# Save
final.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'Shape: {final.shape}')
print(f'\nFinal null summary:')
print(final.isnull().sum().to_string())

Saved to: data/dws_test.csv
Shape: (200, 69)

Final null summary:
Latitude                                 0
Longitude                                0
Sample Date                             96
Total Alkalinity                       200
Electrical Conductance                 200
Dissolved Reactive Phosphorus          200
pet                                      0
_merge_terra                             0
nir                                      0
green                                    0
swir16                                   0
swir22                                   0
NDMI                                     0
MNDWI                                    0
_merge_landsat                           0
Impute_Method                            0
geometry                                 0
STAT_ID                                  0
sc                                       0
ss                                       0
su                                       0
mt                             

## 8. Quick Sanity Checks

In [32]:
# Sanity check: distributions of matched values
check_cols = {
    'Total Alkalinity (DWS)': 'TAL (mg/L CaCO₃)',
    'Electrical Conductance (DWS)': 'EC (µS/cm)',
    'Dissolved Reactive Phosphorus (DWS)': 'DRP (µg/L)',
}

for col, label in check_cols.items():
    if col in final.columns:
        print(f'\n{label}:')
        print(final[col].describe().round(3))


TAL (mg/L CaCO₃):
count    103.000
mean     131.830
std      115.084
min        8.953
25%       54.144
50%       93.910
75%      165.978
max      514.185
Name: Total Alkalinity (DWS), dtype: float64

EC (µS/cm):
count     103.000
mean      622.240
std       758.383
min        99.100
25%       227.750
50%       417.000
75%       644.500
max      5000.000
Name: Electrical Conductance (DWS), dtype: float64

DRP (µg/L):
count     103.000
mean       46.660
std       130.864
min         5.000
25%         5.000
50%        10.000
75%        35.500
max      1156.000
Name: Dissolved Reactive Phosphorus (DWS), dtype: float64


In [33]:
# Check matching quality
print('Spatial matching quality:')
print(f'  Max distance: {final["dist_km"].max():.3f} km')
print(f'  Mean distance: {final["dist_km"].mean():.3f} km')

print(f'\nTemporal matching quality:')
print(f'  Max days diff: {final["days_diff"].max()}')
print(f'  Mean days diff: {final["days_diff"].mean():.1f}')
print(f'  Exact date matches (0 days): {(final["days_diff"] == 0).sum()}')

# Flag potentially poor matches
far_matches = final[final['dist_km'] > 5].shape[0]
old_matches = final[final['days_diff'] > 365].shape[0]
print(f'\n  Rows >5km from nearest station: {far_matches}')
print(f'  Rows >365 days from nearest measurement: {old_matches}')

Spatial matching quality:
  Max distance: 0.000 km
  Mean distance: 0.000 km

Temporal matching quality:
  Max days diff: 63.0
  Mean days diff: 10.1
  Exact date matches (0 days): 10

  Rows >5km from nearest station: 0
  Rows >365 days from nearest measurement: 0


## Unit Conversion Reference

| Parameter | DWS Unit | Competition Unit | Conversion | Column in output |
|-----------|----------|-----------------|------------|------------------|
| TAL (Alkalinity) | mg/L CaCO₃ | mg/L CaCO₃ | None | `Total Alkalinity (DWS)` |
| EC | mS/m | µS/cm | ×10 | `Electrical Conductance (DWS)` |
| PO4-P (DRP) | mg/L | µg/L | ×1000 | `Dissolved Reactive Phosphorus (DWS)` |
| pH | unitless | — | None | `dws_pH` |
| Ca, Mg, Na, Cl, SO4 | mg/L | — | None | `dws_Ca`, `dws_Mg`, etc. |
| P_Tot | mg/L | — | None | `dws_P_Tot` |

In [34]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 69 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   Latitude                             200 non-null    float64       
 1   Longitude                            200 non-null    float64       
 2   Sample Date                          104 non-null    datetime64[ns]
 3   Total Alkalinity                     0 non-null      float64       
 4   Electrical Conductance               0 non-null      float64       
 5   Dissolved Reactive Phosphorus        0 non-null      float64       
 6   pet                                  200 non-null    float64       
 7   _merge_terra                         200 non-null    object        
 8   nir                                  200 non-null    float64       
 9   green                                200 non-null    float64       
 10  swir16        